<a href="https://colab.research.google.com/github/l2Aquel/Superstore_Python_and_Pyspark/blob/main/Pyspark_SuperStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("SuperStore").getOrCreate()

In [3]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("PySpark_SuperStore.csv")
df.show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name     |Segment    |Country      |City           |State         |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                                |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152

In [4]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



  Q1. Convert Order Date and Ship Date to datetime objects and calculate the Shipping Duration (in days) for every order.

In [22]:
from pyspark.sql.functions import to_date,desc,datediff,col

df = df.withColumn('Order Date', to_date(col('Order Date'),'dd-MM-yyyy')) \
       .withColumn('Ship Date', to_date(col('Ship Date'),'dd-MM-yyyy'))

df.withColumn('Shipping Duration',datediff(col('Ship Date'),col('Order Date'))) \
  .select('Order ID','Ship Date','Order Date','Shipping Duration') \
  .show()

+--------------+----------+----------+-----------------+
|      Order ID| Ship Date|Order Date|Shipping Duration|
+--------------+----------+----------+-----------------+
|CA-2016-152156|2016-11-11|2016-11-08|                3|
|CA-2016-152156|2016-11-11|2016-11-08|                3|
|CA-2016-138688|2016-06-16|2016-06-12|                4|
|US-2015-108966|2015-10-18|2015-10-11|                7|
|US-2015-108966|2015-10-18|2015-10-11|                7|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2014-115812|2014-06-14|2014-06-09|                5|
|CA-2017-114412|2017-04-20|2017-04-15|                5|
|CA-2016-161389|2016-12-10|2016-12-05|                5|
|US-2015-118983|2015-11-26|2015

Q2. Which State has the highest total Sales volume, and the lowest total Sales volume?

In [24]:
from pyspark.sql.functions import sum,desc

print("State with Highest Sales: ")
df.groupBy('State').agg(sum('Sales').alias('Total Sales')).orderBy(desc('Total Sales')).limit(1).show()
print("State with Lowest Sales: ")
df.groupBy('State').agg(sum('Sales').alias('Total Sales')).orderBy('Total Sales').limit(1).show()

State with Highest Sales: 
+----------+----------------+
|     State|     Total Sales|
+----------+----------------+
|California|457687.631500001|
+----------+----------------+

State with Lowest Sales: 
+------------+-----------+
|       State|Total Sales|
+------------+-----------+
|North Dakota|     919.91|
+------------+-----------+



Q3. Identify the top 10 unique customers based on the total number of orders placed.

In [20]:
from pyspark.sql.functions import count,desc

df.groupBy('Customer ID','Customer Name').agg(count('Order ID').alias('Total Orders Placed')).orderBy(desc('Total Orders Placed'),'Customer Name').limit(10).show(truncate=False)

+-----------+-------------------+-------------------+
|Customer ID|Customer Name      |Total Orders Placed|
+-----------+-------------------+-------------------+
|WB-21850   |William Brown      |37                 |
|JL-15835   |John Lee           |34                 |
|MA-17560   |Matt Abelman       |34                 |
|PP-18955   |Paul Prost         |34                 |
|CK-12205   |Chloris Kastensmidt|32                 |
|EH-13765   |Edward Hooks       |32                 |
|JD-15895   |Jonathan Doherty   |32                 |
|SV-20365   |Seth Vernon        |32                 |
|AP-10915   |Arthur Prichep     |31                 |
|EP-13915   |Emily Phan         |31                 |
+-----------+-------------------+-------------------+



Q4. Using PySpark, determine the 25th, 50th (median), and 75th percentiles of Sales to understand the distribution of transaction sizes.

In [8]:
df.select('Sales').summary('25%', '50%', '75%').show()

+-------+------+
|summary| Sales|
+-------+------+
|    25%|17.248|
|    50%|54.384|
|    75%|209.94|
+-------+------+



Q5. Create a cross-tabulation (crosstab) showing the count of orders by Region and Category

In [21]:
from pyspark.sql.functions import count

df.groupBy('Region','Category').agg(count('Order ID').alias('Count of Orders')).orderBy('Region','Category').show()

+-------+---------------+---------------+
| Region|       Category|Count of Orders|
+-------+---------------+---------------+
|Central|      Furniture|            481|
|Central|Office Supplies|           1422|
|Central|     Technology|            420|
|   East|      Furniture|            601|
|   East|Office Supplies|           1712|
|   East|     Technology|            535|
|  South|      Furniture|            332|
|  South|Office Supplies|            995|
|  South|     Technology|            293|
|   West|      Furniture|            707|
|   West|Office Supplies|           1897|
|   West|     Technology|            599|
+-------+---------------+---------------+



Q6. Create a 'High Value Order' column where orders with Sales > 500 are flagged as 'Yes', and others as 'No'.

In [10]:
from pyspark.sql.functions import when

df.withColumn('High Value Order',when(df['Sales']>500,'Yes').otherwise('No')).select('Order ID','Sales','High Value Order').show()

+--------------+--------+----------------+
|      Order ID|   Sales|High Value Order|
+--------------+--------+----------------+
|CA-2016-152156|  261.96|              No|
|CA-2016-152156|  731.94|             Yes|
|CA-2016-138688|   14.62|              No|
|US-2015-108966|957.5775|             Yes|
|US-2015-108966|  22.368|              No|
|CA-2014-115812|   48.86|              No|
|CA-2014-115812|    7.28|              No|
|CA-2014-115812| 907.152|             Yes|
|CA-2014-115812|  18.504|              No|
|CA-2014-115812|   114.9|              No|
|CA-2014-115812|1706.184|             Yes|
|CA-2014-115812| 911.424|             Yes|
|CA-2017-114412|  15.552|              No|
|CA-2016-161389| 407.976|              No|
|US-2015-118983|   68.81|              No|
|US-2015-118983|   2.544|              No|
|CA-2014-105893|  665.88|             Yes|
|CA-2014-167164|    55.5|              No|
|CA-2014-143336|    8.56|              No|
|CA-2014-143336|  213.48|              No|
+----------

Q7. Determine which Sub-Category items have a negative profit margin, and calculate the total loss incurred by those sub-categories.

In [26]:
from pyspark.sql.functions import round

loss = df.groupBy('Sub-Category').agg(round(sum('Profit'),2).alias('Total Profit')).where('`Total Profit` < 0')
print('Loss by Sub-Category: ')
loss.show()
print('Overall Loss: ')
loss.agg(round(sum('Total Profit'),2).alias('Total Overall Loss')).show()

Loss by Sub-Category: 
+------------+------------+
|Sub-Category|Total Profit|
+------------+------------+
|    Supplies|     -1189.1|
|   Bookcases|    -3472.56|
|      Tables|   -17725.48|
+------------+------------+

Overall Loss: 
+------------------+
|Total Overall Loss|
+------------------+
|         -22387.14|
+------------------+



Q8. Aggregate total Sales by month and year.

In [12]:
from pyspark.sql.functions import year,month

df.withColumn('Year',year('Order Date')) \
  .withColumn('Month',month('Order Date')) \
  .groupBy('Year','Month') \
  .agg(round(sum('Sales'),2).alias('Total Sales')) \
  .orderBy('Year','Month') \
  .show(48)

+----+-----+-----------+
|Year|Month|Total Sales|
+----+-----+-----------+
|2014|    1|   14236.89|
|2014|    2|    4519.89|
|2014|    3|   55691.01|
|2014|    4|   28295.34|
|2014|    5|   23648.29|
|2014|    6|   34595.13|
|2014|    7|   33946.39|
|2014|    8|   27909.47|
|2014|    9|   81777.35|
|2014|   10|   31453.39|
|2014|   11|   78628.72|
|2014|   12|   69545.62|
|2015|    1|   18174.08|
|2015|    2|   11951.41|
|2015|    3|   38726.25|
|2015|    4|   34195.21|
|2015|    5|   30131.69|
|2015|    6|   24797.29|
|2015|    7|   28765.32|
|2015|    8|   36898.33|
|2015|    9|   64595.92|
|2015|   10|   31404.92|
|2015|   11|   75972.56|
|2015|   12|   74919.52|
|2016|    1|   18542.49|
|2016|    2|   22978.82|
|2016|    3|   51715.88|
|2016|    4|   38750.04|
|2016|    5|   56987.73|
|2016|    6|   40344.53|
|2016|    7|   39261.96|
|2016|    8|   31115.37|
|2016|    9|   73410.02|
|2016|   10|   59687.75|
|2016|   11|   79411.97|
|2016|   12|   96999.04|
|2017|    1|   43971.37|


Q9. Calculate the percentage of total orders that fall into each Ship Mode category.

In [30]:
from pyspark.sql.functions import countDistinct
from pyspark.sql.window import Window as W

result = df.groupBy('Ship Mode').agg(countDistinct('Order ID').alias('Total Orders per Ship Mode')).orderBy(desc('Total Orders per Ship Mode'))

grand_total_window = W.partitionBy()

result = result.withColumn('Total Orders',sum('Total Orders per Ship Mode').over(grand_total_window))

result.withColumn('Percentage',round(100 * (result['Total Orders per Ship Mode'] / result['Total Orders']),2)).select('Ship Mode','Total Orders per Ship Mode','Percentage').show()


+--------------+--------------------------+----------+
|     Ship Mode|Total Orders per Ship Mode|Percentage|
+--------------+--------------------------+----------+
|Standard Class|                      2994|     59.77|
|  Second Class|                       964|     19.25|
|   First Class|                       787|     15.71|
|      Same Day|                       264|      5.27|
+--------------+--------------------------+----------+



Q10. Analyze the financial impact of discounting strategies by calculating the numeric correlation between Discount tiers and net Profit

In [14]:
from pyspark.sql.functions import corr

correlation_result = df.stat.corr('Discount', 'Profit')

print(f"Correlation between Discount and Profit: {correlation_result:.4f}")

Correlation between Discount and Profit: -0.2195


Q11. Calculate the percentage growth (or decline) in Sales for each month compared to the previous month.

In [28]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window as W

result =  df.withColumn('Year',year('Order Date')) \
            .withColumn('Month',month('Order Date')) \
            .groupBy('Year','Month') \
            .agg(round(sum('Sales'),2).alias('Total Sales')) \
            .orderBy('Year','Month')

windowSpec = W.orderBy('Year','Month')
result = result.withColumn('Previous Month Sales',lag('Total Sales').over(windowSpec))


result.withColumn('Mom Gwth Pct',round(100 * (result['Total Sales'] - result['Previous Month Sales'])/result['Previous Month Sales'],2)).show(48)

+----+-----+-----------+--------------------+------------+
|Year|Month|Total Sales|Previous Month Sales|Mom Gwth Pct|
+----+-----+-----------+--------------------+------------+
|2014|    1|   14236.89|                NULL|        NULL|
|2014|    2|    4519.89|            14236.89|      -68.25|
|2014|    3|   55691.01|             4519.89|     1132.13|
|2014|    4|   28295.34|            55691.01|      -49.19|
|2014|    5|   23648.29|            28295.34|      -16.42|
|2014|    6|   34595.13|            23648.29|       46.29|
|2014|    7|   33946.39|            34595.13|       -1.88|
|2014|    8|   27909.47|            33946.39|      -17.78|
|2014|    9|   81777.35|            27909.47|      193.01|
|2014|   10|   31453.39|            81777.35|      -61.54|
|2014|   11|   78628.72|            31453.39|      149.98|
|2014|   12|   69545.62|            78628.72|      -11.55|
|2015|    1|   18174.08|            69545.62|      -73.87|
|2015|    2|   11951.41|            18174.08|      -34.2

Q12. Calculate the 3-month rolling average of Sales for the entire store to smooth out seasonal volatility

In [29]:
from pyspark.sql.functions import avg
from pyspark.sql.window import Window as W

result = df.withColumn('Year',year('Order Date')) \
            .withColumn('Month',month('Order Date')) \
            .groupBy('Year','Month') \
            .agg(round(sum('Sales'),2).alias('Total Sales')) \
            .orderBy('Year','Month')

windowSpec = W.orderBy('Year','Month').rowsBetween(-2,0)
result.withColumn('Moving avg 3 Months',round(avg('Total Sales').over(windowSpec),2)).show(48)

+----+-----+-----------+-------------------+
|Year|Month|Total Sales|Moving avg 3 Months|
+----+-----+-----------+-------------------+
|2014|    1|   14236.89|           14236.89|
|2014|    2|    4519.89|            9378.39|
|2014|    3|   55691.01|           24815.93|
|2014|    4|   28295.34|           29502.08|
|2014|    5|   23648.29|           35878.21|
|2014|    6|   34595.13|           28846.25|
|2014|    7|   33946.39|           30729.94|
|2014|    8|   27909.47|           32150.33|
|2014|    9|   81777.35|           47877.74|
|2014|   10|   31453.39|           47046.74|
|2014|   11|   78628.72|           63953.15|
|2014|   12|   69545.62|           59875.91|
|2015|    1|   18174.08|           55449.47|
|2015|    2|   11951.41|            33223.7|
|2015|    3|   38726.25|           22950.58|
|2015|    4|   34195.21|           28290.96|
|2015|    5|   30131.69|           34351.05|
|2015|    6|   24797.29|           29708.06|
|2015|    7|   28765.32|            27898.1|
|2015|    

Q13. Create a new column showing the cumulative running total of Sales over time (sorted by Order Date).

In [17]:
from pyspark.sql.functions import sum
from pyspark.sql.window import Window as W

windowSpec = W.orderBy('Order Date').rowsBetween(W.unboundedPreceding, W.currentRow)
df.orderBy('Order Date').withColumn('Running Total',round(sum('Sales').over(windowSpec),2)).select('Order Date','Sales','Running Total').show()

+----------+-------+-------------+
|Order Date|  Sales|Running Total|
+----------+-------+-------------+
|2014-01-03| 16.448|        16.45|
|2014-01-04| 11.784|        28.23|
|2014-01-04|272.736|       300.97|
|2014-01-04|   3.54|       304.51|
|2014-01-05| 19.536|       324.04|
|2014-01-06|  19.44|       343.48|
|2014-01-06|  12.78|       356.26|
|2014-01-06|2573.82|      2930.08|
|2014-01-06| 609.98|      3540.06|
|2014-01-06|   5.48|      3545.54|
|2014-01-06| 391.98|      3937.52|
|2014-01-06| 755.96|      4693.48|
|2014-01-06|  31.12|       4724.6|
|2014-01-06|   6.54|      4731.14|
|2014-01-07| 76.728|      4807.87|
|2014-01-07|  10.43|       4818.3|
|2014-01-09|  9.344|      4827.65|
|2014-01-09|   31.2|      4858.85|
|2014-01-10|   2.89|      4861.74|
|2014-01-10|  51.94|      4913.68|
+----------+-------+-------------+
only showing top 20 rows


Q14. Top Products by Category: For each Category, identify the top 3 Product Name items based on Profit

In [18]:
from pyspark.sql.functions import when,dense_rank
from pyspark.sql.window import Window as W

product_profit = df.groupBy('Category','Product Name').agg(round(sum('Profit'),2).alias('Profit'))
windowSpec = W.partitionBy('Category').orderBy(desc('Profit'))
product_profit.withColumn('rank',dense_rank().over(windowSpec)).orderBy('Category','rank').filter('rank <= 3').select('Category','Product Name','Profit','rank').show(truncate=False)

+---------------+---------------------------------------------------------------------------+--------+----+
|Category       |Product Name                                                               |Profit  |rank|
+---------------+---------------------------------------------------------------------------+--------+----+
|Furniture      |Hon Deluxe Fabric Upholstered Stacking Chairs; Rounded Back                |1927.44 |1   |
|Furniture      |Global Deluxe High-Back Manager's Chair                                    |1558.59 |2   |
|Furniture      |Hon Pagoda Stacking Chairs                                                 |1540.7  |3   |
|Office Supplies|Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind|7753.04 |1   |
|Office Supplies|Ibico EPK-21 Electric Binding System                                       |3345.28 |2   |
|Office Supplies|Honeywell Enviracaire Portable HEPA Air Cleaner for 17' x 22' Room         |3247.02 |3   |
|Technology     |Canon image

Q15. Classify your customers based on their total Sales using the following logic:

High: Sales 10,000

Medium: Sales between 2,000 and 10,000

Low: Sales < $2,000

And find the Average Sale and Average Profit of thier Revenue Tiers

In [19]:
from pyspark.sql.functions import when

customers = df.groupBy('Customer ID','Customer Name').agg(round(sum('Sales'),2).alias('Sales'),round(sum('Profit'),2).alias('Profit'))
customers = customers.withColumn('Revenue Tier',when(customers['Sales'] > 10000,'High').when((customers['Sales'] > 2000) & (customers['Sales'] < 10000),'Medium').otherwise('Low'))
customers.groupBy('Revenue Tier').agg(round(avg('Sales'),2).alias('Avg Sale'),round(avg('Profit'),2).alias('Avg Profit')).show()

+------------+--------+----------+
|Revenue Tier|Avg Sale|Avg Profit|
+------------+--------+----------+
|        High|13245.48|    2776.8|
|         Low| 1028.81|    104.39|
|      Medium| 3974.09|    461.93|
+------------+--------+----------+

